<a href="https://colab.research.google.com/github/SZ2012-8CYL1ND3R/symmetrical-system/blob/main/Agentic_AI_Day_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Simulating a long conversation history
fake_history = [
    {"role": "system", "content": "You are Nova."},
    {"role": "user", "content": "Hello."},
    {"role": "assistant", "content": "Hi there."},
    {"role": "user", "content": "What is the weather?"},
    {"role": "assistant", "content": "It's 28°C."},
    {"role": "user", "content": "Tell me a joke."},
    {"role": "assistant", "content": "Why did the coder cross the road?"}
]

# Keep system prompt (index 0) and the last 4 messages
system_prompt = fake_history[0]
recent_messages = fake_history[-4:]

managed_history = [system_prompt] + recent_messages
print(f"Managed History Length: {len(managed_history)}")

Managed History Length: 5


In [8]:
def get_weather(city):
  return f"The weather in {city} is bright and 28°C."

def get_report(car):
  return f"Your {car} is currently sitting idle."

tool_dispatcher = {
    "weather": get_weather,
    "report": get_report
}

conversation_history = [
    {"role": "system", "content": "You are Nova, an interactive multi-tool autonomous agent with memory management."}
]

print("🤖 Nova Agent Initialized with Sliding Memory Window.")
print("Type 'exit' or 'quit' at any time to end the session.\n")

# Interactive Chat Loop with Memory Trimming
while True:
  user_input = input("USER: ")

  if user_input.lower() in ["exit", "quit"]:
    print("ASSISTANT: Goodbye, Salman. Have a wonderful day.")
    break

  # Log user input
  conversation_history.append({"role": "user", "content": user_input})

  # --- MEMORY MANAGEMENT (sliding window ---
  # If history gets longer than 6 messages (exluding the system prompt), trim it.
  if len(conversation_history) > 7:
    system_prompt = conversation_history[0]
    recent_messages = conversation_history[-6:] # Keep the last 6 messages
    conversation_history = [system_prompt] + recent_messages
    print("⚙️ [System: Memory trimmed to keep agent fast.]")

  lower_input = user_input.lower()
  words = user_input.split()

  triggered_tools = []
  tool_results = []

  if "weather" in lower_input:
    city = "Toronto"
    for preposition in ["in", "at"]:
      if preposition in [w.lower() for w in words]:
        lower_words = [w.lower() for w in words]
        idx = lower_words.index(preposition)
        if idx + 1 < len(words):
          city = words[idx + 1].strip("?.,!")
    triggered_tools.append({"tool": "weather", "arg": city})

  if "status" in lower_input or "report" in lower_input or "car" in lower_input:
    triggered_tools.append({"tool": "report", "arg": "Mercedes"})

  if not triggered_tools:
    fallback_message = "I'm sorry, I couldn't figure out which tool to use for that request. Try asking about the weather or the status of your car."
    conversation_history.append({"role": "assistant", "content": fallback_message})
    print(f"ASSISTANT: {fallback_message}\n")
    continue

  for item in triggered_tools:
    t_name = item["tool"]
    t_arg = item["arg"]

    if t_name in tool_dispatcher:
      func = tool_dispatcher[t_name]
      result = func(t_arg)
      tool_results.append(result)
    else:
      tool_results.append(f"Error: Tool '{t_name}' not found.")

  final_output = " | ".join(tool_results)

  conversation_history.append({"role": "assistant", "content": final_output})
  print(f"ASSISTANT: {final_output}\n")

🤖 Nova Agent Initialized with Sliding Memory Window.
Type 'exit' or 'quit' at any time to end the session.

USER: What is the status of my car?
ASSISTANT: Your Mercedes is currently sitting idle.

USER: How is the weather in Mississauga?
ASSISTANT: The weather in Mississauga is bright and 28°C.

USER: How to make cookies?
ASSISTANT: I'm sorry, I couldn't figure out which tool to use for that request. Try asking about the weather or the status of your car.

USER: What is the definition of car?
⚙️ [System: Memory trimmed to keep agent fast.]
ASSISTANT: Your Mercedes is currently sitting idle.

USER: what is the definition of a vehicular device?
⚙️ [System: Memory trimmed to keep agent fast.]
ASSISTANT: I'm sorry, I couldn't figure out which tool to use for that request. Try asking about the weather or the status of your car.

USER: exit
ASSISTANT: Goodbye, Salman. Have a wonderful day.
